# DSA Prefill CP 矩阵拆分可视化

本 notebook 直接调用 SGLang 项目里的拆分函数，可视化 **round-robin-split** 与 **in-seq-split** 两种模式如何切分矩阵，覆盖 hidden_states / KV cache latent / input_ids / position / seq_lens 全部拆分点。

核心函数（均为项目源码，非复刻）：

| 拆分对象 | 函数 | 源码 | 章节 |
|---|---|---|---|
| hidden_states (round-robin) | `dsa_cp_round_robin_split_data` | [dsa/utils.py:98](../python/sglang/srt/layers/attention/dsa/utils.py#L98) | §2 |
| hidden_states (in-seq) | `cp_split_and_rebuild_data` | [cp_utils.py:145](../python/sglang/srt/layers/utils/cp_utils.py#L145) | §3 |
| CP metadata | `prepare_context_parallel_metadata` | [cp_utils.py:491](../python/sglang/srt/layers/utils/cp_utils.py#L491) | §3 |
| AllGather+rerange | `cp_all_gather_rerange_output` | [cp_utils.py:310](../python/sglang/srt/layers/utils/cp_utils.py#L310) | §2/§3 |
| KV cache latent (MLA) | `rebuild_cp_kv_cache` | [deepseek_v2.py:1874](../python/sglang/srt/models/deepseek_v2.py#L1874) | §5 |
| MLP 前后通信 | `dsa_cp_gather/reduce_scatter_hidden_states` | [communicator_dsa_cp.py:55](../python/sglang/srt/layers/communicator_dsa_cp.py#L55) | §6 |
| in-seq Q 分段 | `cp_attn_forward_extend` | [cp_utils.py:449](../python/sglang/srt/layers/utils/cp_utils.py#L449) | §7 |
| input_ids 重排 | `cp_round_robin_input_ids` | [cp_utils.py:191](../python/sglang/srt/layers/utils/cp_utils.py#L191) | §8 |
| position 拆分 | `cp_split_and_rebuild_position` | [cp_utils.py:167](../python/sglang/srt/layers/utils/cp_utils.py#L167) | §9 |
| seq_lens 切分 | `dsa_cp_round_robin_split_q_seqs_cpu` | [dsa/utils.py:221](../python/sglang/srt/layers/attention/dsa/utils.py#L221) | §10 |

> 依赖：只需 CPU + torch，无需 GPU/分布式。通过 monkeypatch `get_attention_cp_rank/size` 和全局 `ServerArgs` 绕过分布式初始化；AllGather 用 concat 模拟（rerange 逻辑与源码一致）。先看 §0.5 全流程坐标，再读各节细节。

## 0. 环境准备：绕过分布式初始化

三个绕过点：
1. `get_global_server_args()` — 用 `object.__new__` 跳过 `ServerArgs.__post_init__`，手设 `enable_dsa_prefill_context_parallel` / `dsa_prefill_cp_mode` 两个字段即可。
2. `get_attention_cp_size/rank` — 这俩在 `dsa/utils.py` 和 `cp_utils.py` 里是 `from ... import` 进来的绑定，需逐模块 patch。
3. AllGather — 真实路径调 NCCL `attn_cp_all_gather_into_tensor`，单进程跑不了；用 concat 模拟通信结果，后续 rerange 严格照搬源码。

> **CP_RANK 切换机制**：cell-2 定义 `CP_RANK=0` 模块级变量，`_fake_cp_rank` 闭包捕获它。后续章节用 `global CP_RANK; CP_RANK = r` 切 rank，闭包读到新值，`dsa_utils`/`cp_utils`/`dp_att` 三个模块的 `get_attention_cp_rank()` 同步生效。改一个变量即切 rank。

## 0.5 全流程坐标：拆分点在 forward 中的位置

下表把 10 个拆分点按 forward 执行顺序排列，先建全局观再读各节细节。

| Forward 步骤 | 拆分点 | 函数 | 章节 |
|---|---|---|---|
| embedding 前 | input_ids CP 重排 | `cp_round_robin_input_ids` | §8 |
| embedding 后, 层循环前 | hidden_states CP split | `dsa_cp_round_robin_split_data` / `cp_split_and_rebuild_data` | §2 / §3 |
| 每层 attention 内 | KV cache latent AllGather+rerange (MLA) | `rebuild_cp_kv_cache` | §5 |
| 每层 attention 内 (in-seq) | Q 分 prev/next 两段 | `cp_attn_forward_extend` | §7 |
| 每层 MLP 前 (round-robin) | hidden_states AllGather | `dsa_cp_gather_hidden_states` | §6 |
| 每层 MLP 后 (round-robin) | hidden_states ReduceScatter | `dsa_cp_reduce_scatter_hidden_states` | §6 |
| 最后一层后 | hidden_states AllGather+rerange | `cp_all_gather_rerange_output` | §2 / §3 |
| (辅助) position 拆分 | `cp_split_and_rebuild_position` | §9 |
| (辅助) 多 seq seq_lens 切分 | `dsa_cp_round_robin_split_q_seqs_cpu` | §10 |

> **round-robin**：attention 整段算（不分 prev/next），MLP 前后加 gather/scatter。
> **in-seq**：attention 分 prev/next 两段算（不同 KV 长度），MLP 前后旁路（走 DeepEP A2A）。

In [1]:
import dataclasses
import torch

# macOS/CPU 无 triton：torch.compile 装饰器（deep_gemm.py:66）import 时触发 triton 加载会炸。
# 在 import sglang 前把 torch.compile 降级为 no-op 绕过。
torch._dynamo.config.disable = True
torch.compile = lambda fn=None, **kw: (fn if fn is not None else (lambda f: f))

import sglang.srt.layers.dp_attention as dp_att
import sglang.srt.layers.attention.dsa.utils as dsa_utils
import sglang.srt.layers.utils.cp_utils as cp_utils
from sglang.srt.server_args import ServerArgs, set_global_server_args_for_scheduler, get_global_server_args

# ---- 1. 伪造全局 ServerArgs（跳过 __post_init__ 的重校验）----
def make_server_args(mode: str) -> ServerArgs:
    args = ServerArgs.__new__(ServerArgs)  # 跳过 dataclass __init__ + __post_init__
    for f in dataclasses.fields(ServerArgs):
        setattr(args, f.name, f.default if f.default is not dataclasses.MISSING else None)
    args.enable_dsa_prefill_context_parallel = True
    args.dsa_prefill_cp_mode = mode          # "round-robin-split" / "in-seq-split"
    args.enable_prefill_context_parallel = (mode == "in-seq-split")
    args.prefill_cp_mode = mode
    return args

def set_mode(mode: str):
    set_global_server_args_for_scheduler(make_server_args(mode))

# ---- 2. 伪造 CP rank/size（闭包捕获全局变量，改 CP_RANK 即切 rank）----
CP_SIZE = 4
CP_RANK = 0
def _fake_cp_size(): return CP_SIZE
def _fake_cp_rank(): return CP_RANK
for mod in (dp_att, dsa_utils, cp_utils):
    mod.get_attention_cp_size = _fake_cp_size
    mod.get_attention_cp_rank = _fake_cp_rank

set_mode("round-robin-split")  # 默认模式
print(f"CP_SIZE={CP_SIZE}  (改 CP_RANK 全局变量即可切 rank)")
print(f"dsa_prefill_cp_mode -> {get_global_server_args().dsa_prefill_cp_mode}")
print(f"is_dsa_prefill_cp_round_robin_split() -> {dsa_utils.is_dsa_prefill_cp_round_robin_split()}")

/Users/user/Downloads/sglang/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CP_SIZE=4  (改 CP_RANK 全局变量即可切 rank)
dsa_prefill_cp_mode -> round-robin-split
is_dsa_prefill_cp_round_robin_split() -> True


/Users/user/Downloads/sglang/python/sglang/srt/layers/attention/fla/utils.py:223: UserWarning: Triton is not supported on current platform, roll back to CPU.
  warnings.warn(


## 1. 构造输入矩阵

用 `seq=16, hidden=4` 的矩阵，每行填入自己的 token id（`row[i] = [i,i,i,i]`），这样拆分后看数字就知道每个 token 去了哪。

In [2]:
# === 真实场景模拟 ===
# 真实流程:
#   input_ids (整数 token id)  ─┐
#                              ├─> embed_tokens 查表 ─> hidden_states (浮点 embedding)
#   input_ids 始终独立保留       ─┘   hidden_states 是无意义浮点, 系统靠 input_ids 追踪 token
#
# 本 notebook 用固定 seed 造"像真"的 hidden_states (随机浮点), input_ids 独立保留。
# 追踪 token 归属: 拆分时 input_ids 同步拆分, 两者位置对应; 不从 hidden 反推。

import torch
torch.manual_seed(0)

SEQ_LEN = 16
HIDDEN = 4

# 1) input_ids: 整数 token id 张量 (始终独立保留)
input_ids = torch.arange(SEQ_LEN, dtype=torch.int32)
print(f"input_ids: {input_ids.tolist()}")
print(f"  (用户请求 tokenize 后得到, 始终在 forward_batch 独立保留, 用来追踪 token)")

# 2) hidden_states: embed_tokens(input_ids) 输出, 随机浮点 (像真 embedding, 无规律)
hidden_states = torch.randn(SEQ_LEN, HIDDEN, dtype=torch.float32)
matrix = hidden_states   # alias, 后续章节用 matrix 名字切分

print(f"\nhidden_states shape={tuple(matrix.shape)} (随机浮点, 像真 embedding):")
print(matrix)
print(f"  (真实场景: 学到的 embedding 权重, 无意义浮点; 拆分靠位置对应 input_ids, 不反推 token id)")

input_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  (用户请求 tokenize 后得到, 始终在 forward_batch 独立保留, 用来追踪 token)

hidden_states shape=(16, 4) (随机浮点, 像真 embedding):
tensor([[-1.1258, -1.1524, -0.2506, -0.4339],
        [ 0.8487,  0.6920, -0.3160, -2.1152],
        [ 0.3223, -1.2633,  0.3500,  0.3081],
        [ 0.1198,  1.2377,  1.1168, -0.2473],
        [-1.3527, -1.6959,  0.5667,  0.7935],
        [ 0.5988, -1.5551, -0.3414,  1.8530],
        [ 0.7502, -0.5855, -0.1734,  0.1835],
        [ 1.3894,  1.5863,  0.9463, -0.8437],
        [-0.6136,  0.0316, -0.4927,  0.2484],
        [ 0.4397,  0.1124,  0.6408,  0.4412],
        [-0.1023,  0.7924, -0.2897,  0.0525],
        [ 0.5229,  2.3022, -1.4689, -1.5867],
        [-0.6731,  0.8728,  1.0554,  0.1778],
        [-0.2303, -0.3918,  0.5433, -0.3952],
        [-0.4462,  0.7440,  1.5210,  3.4105],
        [-1.5312, -1.2341,  1.8197, -0.5515]])
  (真实场景: 学到的 embedding 权重, 无意义浮点; 拆分靠位置对应 input_ids, 不反推 token id)


## 1.5 demo 规模 vs 真模型

为可读性，本 notebook 用缩小版参数。对照真模型（DeepSeek V3, cp_size=8）：

| 量 | demo | 真模型 | 说明 |
|---|---|---|---|
| seq_len | 16 | 8192 | 单 prefill 序列长度 |
| cp_size | 4 | 8 | CP rank 数 |
| 每 rank token (split 态) | 4 | 1024 | seq/cp_size |
| hidden_size | 4 | 7168 | 隐藏维度 |
| kv_lora_rank | 8 | 512 | MLA 压缩 K 维 |
| qk_rope_head_dim | 2 | 64 | rope 维 |
| KV cache latent 维 | 10 | 576 | kv_lora+rope, 不展开 |
| 若展开 (不 absorb) | — | 40960 | 128 heads × 320, 不存 |
| in-seq block 数 | 8 | 16 | 2*cp_size |

> demo 用小数字方便追踪 token id；真模型比例一致（如 split 态每 rank token = seq/cp）。

---
## 2. round-robin-split 模式

源码 [dsa/utils.py:98](../python/sglang/srt/layers/attention/dsa/utils.py#L98)：`input_.view(-1, cp_size, H)[:, cp_rank]`

即把 token 按 `token_idx % cp_size` 轮流分给各 rank。doc 里的图：
```
rank0: token0, token4, token8, token12   (idx % 4 == 0)
rank1: token1, token5, token9, token13   (idx % 4 == 1)
rank2: token2, token6, token10, token14
rank3: token3, token7, token11, token15
```

In [3]:
set_mode("round-robin-split")

print(f"=== round-robin-split, cp_size={CP_SIZE}, seq={SEQ_LEN} ===\n")
print(f"原始 input_ids: {input_ids.tolist()}")
print()

# 同步拆分 hidden_states 和 input_ids (真实流程两者并行切, 位置一一对应)
rank_splits = {}       # hidden_states 切分
input_splits = {}      # input_ids 切分 (同步)
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    # 切 hidden_states (真实代码)
    rank_splits[r] = dsa_utils.dsa_cp_round_robin_split_data(matrix)
    # 注: V2 路径 (GLM 5.2 DSA) 真实只切 hidden_states + positions (deepseek_v2.py:2382-2383),
    # 不切 input_ids; 这里同步切 input_ids 仅为展示 token 归属, 非真实 V2 调用
    input_splits[r] = dsa_utils.dsa_cp_round_robin_split_data(input_ids)
    print(f"rank {r}  hidden shape={tuple(rank_splits[r].shape)}  input_ids={input_splits[r].tolist()}")

print()
print("每 rank 拿到 seq/cp_size = 16/4 = 4 个 token, hidden 和 input_ids 同步切, 位置对应。")
print("→ V2 真实只切 hidden_states; input_ids 同步切是演示取巧, 为展示 token 归属。")

=== round-robin-split, cp_size=4, seq=16 ===

原始 input_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

rank 0  hidden shape=(4, 4)  input_ids=[0, 4, 8, 12]
rank 1  hidden shape=(4, 4)  input_ids=[1, 5, 9, 13]
rank 2  hidden shape=(4, 4)  input_ids=[2, 6, 10, 14]
rank 3  hidden shape=(4, 4)  input_ids=[3, 7, 11, 15]

每 rank 拿到 seq/cp_size = 16/4 = 4 个 token, hidden 和 input_ids 同步切, 位置对应。
→ V2 真实只切 hidden_states; input_ids 同步切是演示取巧, 为展示 token 归属。


### round-robin 的 AllGather + rerange（模拟）

源码 [cp_utils.py:341-358](../python/sglang/srt/layers/utils/cp_utils.py#L341-L358)：
```python
output_tensor = input_tensor.new_empty((input_.shape[0] * cp_size, *input_.shape[1:]))
attn_cp_all_gather_into_tensor(output_tensor, input_tensor)   # NCCL — 模拟为 concat
output_tensor = output_tensor.view(cp_size, -1, *out_shape[1:]).transpose(0, 1).reshape(out_shape)
```

AllGather 后 dim0 顺序 = `[rank0_tokens, rank1_tokens, ...]`。`view(cp_size, -1, H)` 把 dim0 拆成 (rank, per_rank_len)；`transpose(0,1)` → (per_rank_len, rank)；`reshape` 拍平 → token0, token1, ... 恢复正序。

In [4]:
# 模拟 all-gather: concat 各 rank 的局部 hidden (V2 真实只 AllGather hidden, input_ids 不参与通信)
# input_ids 同步 rerange 仅为展示 token 顺序, 非真实 V2 流程
gathered = torch.cat([rank_splits[r] for r in range(CP_SIZE)], dim=0)
gathered_ids = torch.cat([input_splits[r] for r in range(CP_SIZE)], dim=0)
print(f"AllGather 后 (rerange 前) shape={tuple(gathered.shape)}")
print(f"  token id 顺序 = {gathered_ids.tolist()}  (= rank0|rank1|rank2|rank3 的 input_ids)")
print()

# 严格复刻源码的 view-transpose-reshape rerange (同时作用 hidden 和 input_ids)
out_shape = gathered.shape
reranged = gathered.view(CP_SIZE, -1, *out_shape[1:]).transpose(0, 1).reshape(out_shape)
reranged_ids = gathered_ids.view(CP_SIZE, -1).transpose(0, 1).reshape(-1)
print(f"rerange 后 shape={tuple(reranged.shape)}")
print(f"  token id 顺序 = {reranged_ids.tolist()}")
print()
print(f"hidden 恢复正序? {torch.equal(reranged, matrix)}")
print(f"input_ids 恢复正序? {torch.equal(reranged_ids, input_ids)}")

AllGather 后 (rerange 前) shape=(16, 4)
  token id 顺序 = [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]  (= rank0|rank1|rank2|rank3 的 input_ids)

rerange 后 shape=(16, 4)
  token id 顺序 = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

hidden 恢复正序? True
input_ids 恢复正序? True


In [5]:
# 可视化: token -> rank 映射 (来自同步切分的 input_ids)
print("token -> rank 映射 (round-robin):\n")
for t in range(SEQ_LEN):
    r = t % CP_SIZE
    print(f"  token {t:2d} -> rank {r}")

print("\n各 rank 持有的 token id (来自同步切分的 input_ids):\n")
for r in range(CP_SIZE):
    print(f"rank {r}: {input_splits[r].tolist()}")

# 矩阵视图 (与 §3 cell-16 风格统一): token x rank
print("\nround-robin token x rank 矩阵:\n")
header = "       " + "  ".join(f"t{t:2d}" for t in range(SEQ_LEN))
print(header)
for r in range(CP_SIZE):
    row = [f"r{r}" if t % CP_SIZE == r else " ." for t in range(SEQ_LEN)]
    print(f"rank {r}:", "  ".join(row))
print("\n每列恰好一个 rank 标记 -> token t 归 rank t%cp_size。")

token -> rank 映射 (round-robin):

  token  0 -> rank 0
  token  1 -> rank 1
  token  2 -> rank 2
  token  3 -> rank 3
  token  4 -> rank 0
  token  5 -> rank 1
  token  6 -> rank 2
  token  7 -> rank 3
  token  8 -> rank 0
  token  9 -> rank 1
  token 10 -> rank 2
  token 11 -> rank 3
  token 12 -> rank 0
  token 13 -> rank 1
  token 14 -> rank 2
  token 15 -> rank 3

各 rank 持有的 token id (来自同步切分的 input_ids):

rank 0: [0, 4, 8, 12]
rank 1: [1, 5, 9, 13]
rank 2: [2, 6, 10, 14]
rank 3: [3, 7, 11, 15]

round-robin token x rank 矩阵:

       t 0  t 1  t 2  t 3  t 4  t 5  t 6  t 7  t 8  t 9  t10  t11  t12  t13  t14  t15
rank 0: r0   .   .   .  r0   .   .   .  r0   .   .   .  r0   .   .   .
rank 1:  .  r1   .   .   .  r1   .   .   .  r1   .   .   .  r1   .   .
rank 2:  .   .  r2   .   .   .  r2   .   .   .  r2   .   .   .  r2   .
rank 3:  .   .   .  r3   .   .   .  r3   .   .   .  r3   .   .   .  r3

每列恰好一个 rank 标记 -> token t 归 rank t%cp_size。


---
## 3. in-seq-split 模式（zigzag）

把序列切成 `2 * cp_size` 个连续 block，rank r 拿 **block_r 和 block_{2*cp-1-r}**（首尾配对，负载均衡）。

先看 `prepare_context_parallel_metadata` 生成的索引：`split_list` / `zigzag_index` / `cp_reverse_index`。

In [6]:
set_mode("in-seq-split")

cp_segment_num = CP_SIZE * 2   # 8 blocks
print(f"=== in-seq-split, cp_size={CP_SIZE}, seq={SEQ_LEN}, blocks={cp_segment_num} ===\n")

# 每块大小（seq=16, 8 块 → 每块 2 token）
block_size = SEQ_LEN // cp_segment_num
print(f"每 block 大小 = {block_size} token  (block_i = token[i*{block_size}:(i+1)*{block_size}])")
print()

metas = {}
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    meta = cp_utils.prepare_context_parallel_metadata(
        kv_len=SEQ_LEN, cp_rank=r, cp_size=CP_SIZE,
        seqs_len=[SEQ_LEN], extend_seqs_len=[SEQ_LEN], device="cpu",
    )
    metas[r] = meta
    print(f"rank {r}:")
    print(f"  zigzag_index   = {meta.zigzag_index}   (取 split_list 的哪些块)")
    print(f"  per_rank_token = {meta.per_rank_actual_token[r]}  (本 rank 实际 token 数)")
    print(f"  → 拿 block {meta.zigzag_index[0]} + block {meta.zigzag_index[1]}")
print()
print(f"split_list (全 block 大小) = {metas[0].split_list}")
print(f"cp_reverse_index (allgather 后重排) = {metas[0].cp_reverse_index}")

=== in-seq-split, cp_size=4, seq=16, blocks=8 ===

每 block 大小 = 2 token  (block_i = token[i*2:(i+1)*2])

rank 0:
  zigzag_index   = [0, 7]   (取 split_list 的哪些块)
  per_rank_token = 4  (本 rank 实际 token 数)
  → 拿 block 0 + block 7
rank 1:
  zigzag_index   = [1, 6]   (取 split_list 的哪些块)
  per_rank_token = 4  (本 rank 实际 token 数)
  → 拿 block 1 + block 6
rank 2:
  zigzag_index   = [2, 5]   (取 split_list 的哪些块)
  per_rank_token = 4  (本 rank 实际 token 数)
  → 拿 block 2 + block 5
rank 3:
  zigzag_index   = [3, 4]   (取 split_list 的哪些块)
  per_rank_token = 4  (本 rank 实际 token 数)
  → 拿 block 3 + block 4

split_list (全 block 大小) = [2, 2, 2, 2, 2, 2, 2, 2]
cp_reverse_index (allgather 后重排) = [0, 2, 4, 6, 7, 5, 3, 1]


### 调用 `cp_split_and_rebuild_data` 执行真实切分

该函数 [cp_utils.py:145](../python/sglang/srt/layers/utils/cp_utils.py#L145) 读 `forward_batch.attn_cp_metadata`，按 `split_list` 切全序列，再按 `zigzag_index` 重排成本 rank 的 `[block_r, block_{2cp-1-r}]`。

In [7]:
from types import SimpleNamespace

rank_splits_seq = {}        # hidden_states
input_splits_seq = {}       # input_ids (同步, 用 cp_split_and_rebuild_position 1D 真函数)
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    fake_fb = SimpleNamespace(attn_cp_metadata=metas[r])
    # 切 hidden_states (2D, 用 cp_split_and_rebuild_data)
    rank_splits_seq[r] = cp_utils.cp_split_and_rebuild_data(fake_fb, matrix)
    # 注: V2 路径真实切 position 用此函数 (deepseek_v2.py:2383); input_ids 不切.
    # 这里借 1D 切分函数展示 input_ids 的 zigzag 归属, 非真实 V2 调用
    input_splits_seq[r] = cp_utils.cp_split_and_rebuild_position(fake_fb, input_ids)
    print(f"rank {r}  hidden shape={tuple(rank_splits_seq[r].shape)}  input_ids={input_splits_seq[r].tolist()}")

print()
print("验证 zigzag 配对 (来自同步切分的 input_ids):")
for r in range(CP_SIZE):
    t = input_splits_seq[r].tolist()
    print(f"  rank {r}: block_r={t[:block_size]}  block_{{2cp-1-r}}={t[block_size:]}  (首+尾)")

rank 0  hidden shape=(4, 4)  input_ids=[0, 1, 14, 15]
rank 1  hidden shape=(4, 4)  input_ids=[2, 3, 12, 13]
rank 2  hidden shape=(4, 4)  input_ids=[4, 5, 10, 11]
rank 3  hidden shape=(4, 4)  input_ids=[6, 7, 8, 9]

验证 zigzag 配对 (来自同步切分的 input_ids):
  rank 0: block_r=[0, 1]  block_{2cp-1-r}=[14, 15]  (首+尾)
  rank 1: block_r=[2, 3]  block_{2cp-1-r}=[12, 13]  (首+尾)
  rank 2: block_r=[4, 5]  block_{2cp-1-r}=[10, 11]  (首+尾)
  rank 3: block_r=[6, 7]  block_{2cp-1-r}=[8, 9]  (首+尾)


### in-seq 的 AllGather + rerange（模拟）

源码 [cp_utils.py:361-378](../python/sglang/srt/layers/utils/cp_utils.py#L361-L378)：AllGather 后按 `cp_reverse_index` 重排回正序。

AllGather 输出顺序 = `[rank0_prevs, rank0_nexts, rank1_prevs, rank1_nexts, ...]`（`reverse_split_len` 给出每段长度）。

In [8]:
meta0 = metas[0]

# 模拟 all-gather: 按 reverse_split_len 拼接各 rank 的 [prev, next] 段 hidden
# (V2 真实只 AllGather hidden; input_ids 同步仅为展示 token 顺序, 非真实)
pieces, pieces_ids = [], []
for r in range(CP_SIZE):
    split = rank_splits_seq[r]
    split_ids = input_splits_seq[r]
    prev_len = meta0.per_rank_actual_token[r] // 2
    pieces.append(split[:prev_len]);            pieces_ids.append(split_ids[:prev_len])
    pieces.append(split[prev_len:]);            pieces_ids.append(split_ids[prev_len:])
gathered_seq = torch.cat(pieces, dim=0)
gathered_ids_seq = torch.cat(pieces_ids, dim=0)
print(f"AllGather 后 (rerange 前) token id = {gathered_ids_seq.tolist()}")
print(f"  (= r0.prev | r0.next | r1.prev | r1.next | ...)")
print(f"  reverse_split_len = {meta0.reverse_split_len}")
print()

# 严格复刻源码 rerange: 按 reverse_split_len 切分, 再按 cp_reverse_index 重排
outputs_list = list(torch.split(gathered_seq, meta0.reverse_split_len, dim=0))
outputs_ids_list = list(torch.split(gathered_ids_seq, meta0.reverse_split_len, dim=0))
reranged_seq = torch.cat([outputs_list[i] for i in meta0.cp_reverse_index], dim=0).view(-1, HIDDEN)
reranged_ids_seq = torch.cat([outputs_ids_list[i] for i in meta0.cp_reverse_index], dim=0)
print(f"rerange 后 token id = {reranged_ids_seq.tolist()}")
print(f"hidden 恢复正序? {torch.equal(reranged_seq, matrix)}")
print(f"input_ids 恢复正序? {torch.equal(reranged_ids_seq, input_ids)}")

AllGather 后 (rerange 前) token id = [0, 1, 14, 15, 2, 3, 12, 13, 4, 5, 10, 11, 6, 7, 8, 9]
  (= r0.prev | r0.next | r1.prev | r1.next | ...)
  reverse_split_len = [2, 2, 2, 2, 2, 2, 2, 2]

rerange 后 token id = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
hidden 恢复正序? True
input_ids 恢复正序? True


In [9]:
# 直观图示：8 个 block 在各 rank 间的分配
print("in-seq zigzag block 分配 (cp_size=4, 8 blocks):\n")
print("block:    ", "  ".join(f"b{i}" for i in range(cp_segment_num)))
print("token 范围:", "  ".join(f"{i*block_size}-{i*block_size+block_size-1}" for i in range(cp_segment_num)))
print()
for r in range(CP_SIZE):
    owned = metas[r].zigzag_index
    row = []
    for b in range(cp_segment_num):
        row.append(f"r{r}" if b in owned else "  .")
    print(f"rank {r}:   " + "  ".join(row) + f"   (block {owned[0]} + block {owned[1]})")

print("\n首尾配对: rank r 拿 block_r (前段) + block_{2cp-1-r} (尾段)，负载均衡。")

in-seq zigzag block 分配 (cp_size=4, 8 blocks):

block:     b0  b1  b2  b3  b4  b5  b6  b7
token 范围: 0-1  2-3  4-5  6-7  8-9  10-11  12-13  14-15

rank 0:   r0    .    .    .    .    .    .  r0   (block 0 + block 7)
rank 1:     .  r1    .    .    .    .  r1    .   (block 1 + block 6)
rank 2:     .    .  r2    .    .  r2    .    .   (block 2 + block 5)
rank 3:     .    .    .  r3  r3    .    .    .   (block 3 + block 4)

首尾配对: rank r 拿 block_r (前段) + block_{2cp-1-r} (尾段)，负载均衡。


---
## 4. hidden_states 拆分对比（§2-3 小结）

| | round-robin-split | in-seq-split |
|---|---|---|
| 切分粒度 | 单 token 级 (`idx % cp_size`) | block 级 (`2*cp_size` 个连续 block) |
| rank r 持有 | token `r, r+cp, r+2cp, ...` | block_r + block_{2cp-1-r} (首尾配对) |
| 切分函数 | `dsa_cp_round_robin_split_data` | `cp_split_and_rebuild_data` (用 metadata) |
| AllGather rerange | `view(cp,-1,H).transpose(0,1).reshape` | 按 `cp_reverse_index` 重排 |
| 需要 metadata? | 否（纯 stride 切片） | 是 (`split_list`/`zigzag_index`/`cp_reverse_index`) |
| MoE 后端 | TP-only (none) | DeepEP (EP) |

In [10]:
# 两种模式拆分对比 (token id 来自同步切分的 input_ids, 演示用; V2 真实只切 hidden)
print(f"原始 input_ids: {input_ids.tolist()}\n")

print("round-robin-split (各 rank 的 input_ids):")
for r in range(CP_SIZE):
    print(f"  rank {r}: {input_splits[r].tolist()}")

print("\nin-seq-split (各 rank 的 input_ids):")
for r in range(CP_SIZE):
    print(f"  rank {r}: {input_splits_seq[r].tolist()}")

print("\n两种模式 AllGather+rerange 后 hidden 和 input_ids 均恢复正序 (已验证 torch.equal)。")

原始 input_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

round-robin-split (各 rank 的 input_ids):
  rank 0: [0, 4, 8, 12]
  rank 1: [1, 5, 9, 13]
  rank 2: [2, 6, 10, 14]
  rank 3: [3, 7, 11, 15]

in-seq-split (各 rank 的 input_ids):
  rank 0: [0, 1, 14, 15]
  rank 1: [2, 3, 12, 13]
  rank 2: [4, 5, 10, 11]
  rank 3: [6, 7, 8, 9]

两种模式 AllGather+rerange 后 hidden 和 input_ids 均恢复正序 (已验证 torch.equal)。


---
## 5. MLA 的 KV cache 在 CP 下如何拆分

前面 2-4 节展示的是 **hidden_states / input_ids** 的拆分（`dsa_cp_round_robin_split_data` / `cp_split_and_rebuild_data`）。
MLA attention 层里另有 **KV cache latent** 的 CP 拆分，由 `DeepseekV2AttentionMLA.rebuild_cp_kv_cache` 完成。

**源码位置**：[deepseek_v2.py:1874](../python/sglang/srt/models/deepseek_v2.py#L1874)

```python
def rebuild_cp_kv_cache(self, latent_cache, forward_batch, k_nope, k_pe):
    latent_cache[..., : self.kv_lora_rank] = k_nope.squeeze(1)   # 压缩 K
    latent_cache[..., self.kv_lora_rank :] = k_pe.squeeze(1)     # rope K
    latent_cache_output = cp_all_gather_rerange_output(          # ← CP 通信+rerange
        latent_cache.contiguous(), self.cp_size, forward_batch, torch.cuda.current_stream(),
    )
    k_nope = latent_cache_output[..., : self.kv_lora_rank].unsqueeze(1)
    k_pe = latent_cache_output[..., self.kv_lora_rank :].unsqueeze(1)
    return k_nope, k_pe
```

**调用点**：`forward_absorb_prepare` ([forward_mla.py:384](../python/sglang/srt/models/deepseek_common/attention_forward_methods/forward_mla.py#L384))，条件 `dsa_use_prefill_cp(fb) or mla_use_prefill_cp(fb)`。

**关键点**：KV cache 存的是 **压缩 latent**（`kv_lora_rank + qk_rope_head_dim` 维，例如 512+64=576），**不是** 展开的 `qk_nope/v` 全维。`rebuild_cp_kv_cache` 在这 576 维上做 AllGather+rerange，从不展开。

> CPU 跑通需额外 stub：`torch.cuda.current_stream`、`get_attention_cp_group`、`attn_cp_all_gather_into_tensor`（NCCL 用 concat 模拟）。

In [11]:
from contextlib import contextmanager
from sglang.srt.layers.utils.cp_utils import ContextParallelMetadata
from sglang.srt.models.deepseek_v2 import DeepseekV2AttentionMLA   # ← 真函数来自 deepseek_v2.py
from types import SimpleNamespace

# stub 5 件套: 让 rebuild_cp_kv_cache 在 CPU 跑通 (round-robin 路径只需这 5 个)
torch.cuda.current_stream = lambda: object()
cp_utils.get_attention_cp_group = lambda: None
cp_utils.is_allocation_symmetric = lambda: False
cp_utils.use_symmetric_memory = lambda *a, **k: contextmanager(lambda: (yield))()

# ---- MLA 参数（缩小版：hidden=16, q_lora=8, kv_lora=8, qk_rope=2; 真模型 7168/512/512/64）----
HID, Q_LORA, kv_lora, qk_rope = 16, 8, 8, 2
SEQ = 8
set_mode("round-robin-split")
torch.manual_seed(0)

# 真实流程 (deepseek_v2.py:2382 CP split hidden → forward_mla.py:151 投影):
#   1. hidden_states 已 CP split (round-robin) → 每 rank [seq/cp, D]
#   2. qkv_latent = fused_qkv_a_proj_with_mqa(hidden)  → [seq/cp, q_lora+kv_lora+qk_rope]
#   3. latent_cache = qkv_latent[:, q_lora:]  → [seq/cp, kv_lora+qk_rope]
#   4. k_nope/k_pe 从 latent_cache 取, norm, unsqueeze
#   5. rebuild_cp_kv_cache(latent_cache, fb, k_nope, k_pe): 写回 latent → AllGather → [seq, ...]
hidden_full = torch.randn(SEQ, HID, dtype=torch.float32)
W_a = torch.randn(HID, Q_LORA + kv_lora + qk_rope, dtype=torch.float32)   # 模拟 fused proj 权重

def _rmsnorm(x, eps=1e-6):
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + eps)

# 为每 rank 预算 split hidden + latent_cache (模拟 4 rank 各自的 split 态)
hidden_splits, latent_splits, k_nope_splits, k_pe_splits = {}, {}, {}, {}
global CP_RANK
for r in range(CP_SIZE):
    CP_RANK = r
    # round-robin split hidden (真实: cp_split_and_rebuild_data 在 forward 入口做)
    hidden_splits[r] = dsa_utils.dsa_cp_round_robin_split_data(hidden_full)
    qkv_latent = hidden_splits[r] @ W_a                                   # fused_qkv_a_proj
    q, latent_cache = qkv_latent.split([Q_LORA, kv_lora + qk_rope], dim=-1)
    k_nope = _rmsnorm(latent_cache[..., :kv_lora]).unsqueeze(1)           # kv_a_layernorm + unsqueeze
    k_pe = latent_cache[..., kv_lora:].unsqueeze(1)
    latent_splits[r] = latent_cache.clone()
    k_nope_splits[r], k_pe_splits[r] = k_nope, k_pe

# AllGather stub: 拼接 4 rank 的 latent (真实 NCCL all-gather 各 rank 不同数据)
def _fake_ag(output, input):
    # input = rank 0 的 latent_cache (rebuild 已写入 k_nope/k_pe); 其余 rank 用预存的 latent_splits
    # 真实: 各 rank 的 latent_cache 在 AllGather 时都已写好, 这里用 latent_splits 模拟
    out = torch.cat([latent_splits[r] for r in range(CP_SIZE)], dim=0)
    output.copy_(out)
cp_utils.attn_cp_all_gather_into_tensor = _fake_ag

CP_RANK = 0
print(f"hidden_full {tuple(hidden_full.shape)} → round-robin split → 每 rank hidden {tuple(hidden_splits[0].shape)}")
print(f"  fused_qkv_a_proj → latent_cache (split) {tuple(latent_splits[0].shape)}  (= [seq/cp={SEQ//CP_SIZE}, kv_lora+qk_rope={kv_lora+qk_rope}])")
print(f"  k_nope {tuple(k_nope_splits[0].shape)}, k_pe {tuple(k_pe_splits[0].shape)}")
print(f"  KV cache latent 维度 = {kv_lora + qk_rope} (真模型 576)")
print()

# 构造最小 self, 复用真 rebuild_cp_kv_cache 方法
class FakeMLA:
    kv_lora_rank = kv_lora
    cp_size = CP_SIZE
    rebuild_cp_kv_cache = DeepseekV2AttentionMLA.rebuild_cp_kv_cache

fb = SimpleNamespace(attn_cp_metadata=ContextParallelMetadata())   # round-robin: 空 metadata
# 传 rank 0 的 split latent_cache + k_nope/k_pe (真实流程每 rank 各自调)
k_nope_out, k_pe_out = FakeMLA().rebuild_cp_kv_cache(
    latent_splits[0].clone(), fb, k_nope_splits[0], k_pe_splits[0]
)

print(f"rebuild_cp_kv_cache 后: k_nope {tuple(k_nope_out.shape)}, k_pe {tuple(k_pe_out.shape)}")
print(f"  dim0 = seq = {SEQ}  (split latent [seq/cp] AllGather → [seq], 不是 seq*cp)")
print()
print("→ KV cache 全程压缩 latent 态, 未展开。rebuild_cp_kv_cache 在 576 维上 AllGather+rerange, split→全量。")

/Users/user/Downloads/sglang/python/sglang/srt/layers/quantization/awq/awq.py:52: UserWarning: Only CUDA, HIP and XPU support AWQ currently.
  warnings.warn(f"Only CUDA, HIP and XPU support AWQ currently.")


hidden_full (8, 16) → round-robin split → 每 rank hidden (2, 16)
  fused_qkv_a_proj → latent_cache (split) (2, 10)  (= [seq/cp=2, kv_lora+qk_rope=10])
  k_nope (2, 1, 8), k_pe (2, 1, 2)
  KV cache latent 维度 = 10 (真模型 576)

rebuild_cp_kv_cache 后: k_nope (8, 1, 8), k_pe (8, 1, 2)
  dim0 = seq = 8  (split latent [seq/cp] AllGather → [seq], 不是 seq*cp)

→ KV cache 全程压缩 latent 态, 未展开。rebuild_cp_kv_cache 在 576 维上 AllGather+rerange, split→全量。


/Users/user/Downloads/sglang/python/sglang/srt/layers/quantization/gguf.py:64: UserWarning: Only CUDA, MUSA and NPU support GGUF quantization currently.
  warnings.warn(f"Only CUDA, MUSA and NPU support GGUF quantization currently.")


### 为什么 MLA 不需要展开 KV？weight absorption

MLA 的 K/V 本是低秩分解：`K = W_KC @ k_nope`，`V = W_VC @ k_nope`（k_nope 是压缩 latent）。

源码 `forward_absorb_prepare` ([forward_mla.py:287-371](../python/sglang/srt/models/deepseek_common/attention_forward_methods/forward_mla.py#L287-L371)) 把权重吸收到 Q 侧和输出侧，让 KV cache 只存压缩 latent：

```
常规 (展开):    K_full = k_nope @ W_KC.T    [seq, heads, qk_nope_dim]
               V_full = k_nope @ W_VC.T    [seq, heads, v_dim]
               attn(Q, K_full, V_full)     ← KV cache 存全维，贵

absorb (不展开): q_nope_out = q_nope @ W_KC   [heads, kv_lora]   ← W_KC 吸到 Q 侧
               attn(q_nope_out, k_nope, k_nope)  ← KV cache 只存 k_nope (kv_lora 维)
               out = attn_out @ W_VC       ← W_VC 吸到输出侧
```

所以 KV cache 维度 = `kv_lora_rank + qk_rope` (576)，**不是** `heads * (qk_nope + v)` (~2048)。`rebuild_cp_kv_cache` 在这 576 维上做 CP AllGather+rerange，与第 2 节 round-robin hidden_states 的拆分逻辑同源。

In [12]:
# 对比：KV cache 压缩 latent vs 展开全维
num_heads = 128
qk_nope_dim = 128
v_dim = 128
kv_lora_real = 512
qk_rope_real = 64

compressed = kv_lora_real + qk_rope_real             # 576
expanded = num_heads * (qk_nope_dim + v_dim + qk_rope_real)  # 不存

print("DeepSeek V3 真实参数:")
print(f"  KV cache 压缩 latent = {kv_lora_real} + {qk_rope_real} = {compressed} 维/token")
print(f"  若展开 (不 absorb) = heads({num_heads}) × (qk_nope {qk_nope_dim} + v {v_dim} + rope {qk_rope_real}) = {expanded} 维/token")
print(f"  压缩比 = {expanded}/{compressed} = {expanded/compressed:.1f}x 节省")
print()
print("→ MLA 用 weight absorption 让 KV cache 存压缩 latent (576 维)，")
print("  rebuild_cp_kv_cache 在这 576 维上做 CP AllGather+rerange，不展开。")

DeepSeek V3 真实参数:
  KV cache 压缩 latent = 512 + 64 = 576 维/token
  若展开 (不 absorb) = heads(128) × (qk_nope 128 + v 128 + rope 64) = 40960 维/token
  压缩比 = 40960/576 = 71.1x 节省

→ MLA 用 weight absorption 让 KV cache 存压缩 latent (576 维)，
  rebuild_cp_kv_cache 在这 576 维上做 CP AllGather+rerange，不展开。


---
## 6. MLP 前后 hidden_states 的 gather/scatter（round-robin 专属）

round-robin-split 模式下，每层 MLP 前后需对 hidden_states 做通信：MLP **前** AllGather `[seq/cp, D]→[seq, D]`，MLP **后** ReduceScatter `[seq, D]→[seq/cp, D]`。in-seq 模式旁路。

**源码**：[communicator_dsa_cp.py:55-76](../python/sglang/srt/layers/communicator_dsa_cp.py#L55-L76)

```python
def dsa_cp_gather_hidden_states(hidden_states):           # MLP 前
    hidden_states, local_hidden_states = get_local_dp_buffer(...), hidden_states
    attn_cp_all_gather_into_tensor(hidden_states, local_hidden_states)  # [seq/cp,D]→[seq,D]
    return hidden_states

def dsa_cp_reduce_scatter_hidden_states(hidden_states):   # MLP 后
    hidden_states = hidden_states.tensor_split(cp_size)[cp_rank]         # [seq,D]→[seq/cp,D]
    attn_cp_reduce_scatter_tensor(hidden_states, input_hidden_states)
    return hidden_states
```

**调用点**：`DSACPLayerCommunicator._gather_hidden_states_and_residual` ([communicator_dsa_cp.py:185](../python/sglang/srt/layers/communicator_dsa_cp.py#L185)) 在 `prepare_mlp` 阶段；`_scatter_hidden_states` ([communicator_dsa_cp.py:230](../python/sglang/srt/layers/communicator_dsa_cp.py#L230)) 在 `postprocess_layer` 阶段。条件 `dsa_use_prefill_cp(fb)`。

> 这俩函数依赖 `get_local_dp_buffer`（全局 hidden_size buffer），CPU 跑通需 stub。下面用语义等价的 concat/split 演示通信效果。

In [13]:
# round-robin 模式 MLP 前后 hidden_states 通信
# 真实流程 (deepseek_v2.py:2088-2133):
#   attn 输出 hidden_states (split 态) → prepare_mlp 内 dsa_cp_gather_hidden_states → MLP → postprocess_layer 内 dsa_cp_reduce_scatter
# 所以 MLP 输入 = cell-6 切出的 rank_splits[0] (本 rank split hidden_states), 非独立 arange
set_mode("round-robin-split")

import sglang.srt.layers.communicator_dsa_cp as comm_mod
from contextlib import contextmanager

class _DummyGroup:
    world_size = CP_SIZE
    rank_in_group = CP_RANK
comm_mod.get_attention_dp_size = lambda: 1
comm_mod.get_attention_tp_size = lambda: 1
comm_mod.get_attention_cp_size = lambda: CP_SIZE
comm_mod.get_attention_cp_rank = lambda: CP_RANK
comm_mod.get_attention_cp_group = lambda: _DummyGroup()
# get_local_dp_buffer 是 AllGather 输出 buffer, 全量 [seq, D] (容纳 4 rank 拼接)
comm_mod.get_local_dp_buffer = lambda group: torch.empty(SEQ_LEN, HIDDEN, dtype=torch.float32)
comm_mod.is_allocation_symmetric = lambda: False
comm_mod.use_symmetric_memory = lambda *a, **k: contextmanager(lambda: (yield))()

# stub NCCL: AllGather=concat (各 rank split hidden 拼回全量), ReduceScatter=按 rank 切片
def _ag(output, input):
    # 真实: 各 rank 的 split hidden (rank_splits[r]) AllGather 拼成全量
    # 这里 input = rank_splits[0], 模拟其余 rank 用同源 matrix 切片
    out = torch.cat([rank_splits[r] for r in range(CP_SIZE)], dim=0)
    output.copy_(out)
def _rs(output, input):
    full = input.view(CP_SIZE, -1, *input.shape[1:])
    output.copy_(full[CP_RANK])
comm_mod.attn_cp_all_gather_into_tensor = _ag
comm_mod.attn_cp_reduce_scatter_tensor = _rs

# MLP 输入 = attention 输出的本 rank split hidden_states (复用 cell-6 的 rank_splits[0])
CP_RANK = 0
local_hs = rank_splits[CP_RANK]   # ← 真实: 来自上一阶段 attention 输出, 非独立 arange
print(f"MLP 输入 (rank {CP_RANK} split hidden) shape={tuple(local_hs.shape)}  (来自 cell-6 rank_splits)")
print(f"  对应 token id: {input_splits[CP_RANK].tolist()}")
print(f"  (每 rank 视角: 只有自己的 {SEQ_LEN//CP_SIZE} 个 token)")
print()

# prepare_mlp → dsa_cp_gather_hidden_states: AllGather 拼 4 rank
gathered = comm_mod.dsa_cp_gather_hidden_states(local_hs)
print(f"after gather shape={tuple(gathered.shape)}  (全量 seq={SEQ_LEN})")
print(f"  对应 token id: {torch.cat([input_splits[r] for r in range(CP_SIZE)]).tolist()}")
print()

# postprocess_layer → dsa_cp_reduce_scatter_hidden_states: 切回本 rank
CP_RANK = 0
scattered = comm_mod.dsa_cp_reduce_scatter_hidden_states(gathered)
print(f"after reduce_scatter shape={tuple(scattered.shape)}  (回到 split 态)")
print(f"  对应 token id: {input_splits[CP_RANK].tolist()}")
print()
print("→ round-robin 每层: attn 输出(split) → prepare_mlp gather → MLP → postprocess_layer reduce_scatter → split")
print("  hidden_states 在 gather/scatter 间是全量, 进出 MLP 边界回到 split。in-seq 旁路 (走 DeepEP A2A)。")

MLP 输入 (rank 0 split hidden) shape=(4, 4)  (来自 cell-6 rank_splits)
  对应 token id: [0, 4, 8, 12]
  (每 rank 视角: 只有自己的 4 个 token)

after gather shape=(16, 4)  (全量 seq=16)
  对应 token id: [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]

after reduce_scatter shape=(4, 4)  (回到 split 态)
  对应 token id: [0, 4, 8, 12]

→ round-robin 每层: attn 输出(split) → prepare_mlp gather → MLP → postprocess_layer reduce_scatter → split
  hidden_states 在 gather/scatter 间是全量, 进出 MLP 边界回到 split。in-seq 旁路 (走 DeepEP A2A)。


---
## 7. in-seq attention: Q 分 prev/next 两段（zigzag 闭环）

in-seq-split 模式下，rank r 持 `[block_r, block_{2cp-1-r}]` 两段非连续 token。attention 需分别对前段（prev，KV = blocks[0..r]）和后段（next，KV = blocks[0..2cp-1-r]）计算，再拼接。

**源码**：[cp_utils.py:449](../python/sglang/srt/layers/utils/cp_utils.py#L449)

```python
def cp_attn_forward_extend(forward_batch, q, device, attn_fn):
    cp_meta = forward_batch.attn_cp_metadata
    q_prev = q[: cp_meta.total_q_prev_tokens]          # 前段 Q
    q_next = q[cp_meta.total_q_prev_tokens :]          # 后段 Q
    result_prev = attn_fn(q_prev, cu_seqlens_q_prev, kv_len_prev, max_seqlen_q_prev)
    result_next = attn_fn(q_next, cu_seqlens_q_next, kv_len_next, max_seqlen_q_next)
    return torch.concat([result_prev, result_next], dim=0)
```

**`attn_fn` 签名**：`attn_fn(q, cu_seqlens_q, cache_seqlens, max_seqlen_q) → result`。只有这 4 个 CP 参数在 prev/next 间不同，其余 backend 参数走闭包。

**关键 metadata**（`prepare_context_parallel_metadata` 算出）：
- `total_q_prev_tokens` / `total_q_next_tokens`：prev/next 的 Q token 数（split 点）
- `kv_len_prev` / `kv_len_next`：prev 看到 KV 长度（blocks[0..r]）、next 看到 KV 长度（blocks[0..2cp-1-r]）
- `cu_seqlens_q_prev/next`：FlashAttention 变长 Q 边界 `[bs+1]`

In [14]:
set_mode("in-seq-split")

# 重新生成 in-seq metadata（rank 0）
global CP_RANK
CP_RANK = 0
meta0 = cp_utils.prepare_context_parallel_metadata(
    kv_len=SEQ_LEN, cp_rank=CP_RANK, cp_size=CP_SIZE,
    seqs_len=[SEQ_LEN], extend_seqs_len=[SEQ_LEN], device="cpu",
)
print(f"rank 0 (in-seq) zigzag: block {meta0.zigzag_index[0]} + block {meta0.zigzag_index[1]}")
print(f"  total_q_prev={meta0.total_q_prev_tokens}  total_q_next={meta0.total_q_next_tokens}")
print(f"  actual_seq_q_prev={meta0.actual_seq_q_prev_list}  actual_seq_q_next={meta0.actual_seq_q_next_list}")
print(f"  kv_len_prev={meta0.kv_len_prev_list}  (看到 blocks[0..{CP_RANK}] = {meta0.kv_len_prev_list[0]} token)")
print(f"  kv_len_next={meta0.kv_len_next_list}  (看到 blocks[0..{2*CP_SIZE-1-CP_RANK}] = {meta0.kv_len_next_list[0]} token)")
print()

# 真实 Q: q_b_proj 输出 view(-1, num_local_heads, qk_head_dim) (forward_mla.py:262), 3D
# rank 0 持 total_q_prev+total_q_next 个 token, 布局 [all_prev | all_next]
NUM_HEADS, QK_HEAD_DIM = 4, 6
q = torch.arange((meta0.total_q_prev_tokens + meta0.total_q_next_tokens) * NUM_HEADS * QK_HEAD_DIM,
                 dtype=torch.float32).reshape(-1, NUM_HEADS, QK_HEAD_DIM)
print(f"Q (split, [prev|next]) shape={tuple(q.shape)}  (3D: [tokens, heads, qk_head_dim])")
print(f"  q_prev shape={tuple(q[:meta0.total_q_prev_tokens].shape)}  (block 0 的 Q)")
print(f"  q_next shape={tuple(q[meta0.total_q_prev_tokens:].shape)}  (block 7 的 Q)")
print()

# attn_fn 桩: 标记 prev/next 各跑一次 (真实是 FA kernel, 这里返回 q*10 看分段)
call_count = [0]
def fake_attn(q_seg, cu_seqlens, cache_seqlens, max_seqlen_q):
    call_count[0] += 1
    tag = 'prev' if call_count[0] == 1 else 'next'
    print(f"  attn_fn #{call_count[0]} ({tag}): q_seg shape={tuple(q_seg.shape)}, kv_len={cache_seqlens.tolist()}, max_q={max_seqlen_q}")
    return q_seg * 10

fb = SimpleNamespace(attn_cp_metadata=meta0)
out = cp_utils.cp_attn_forward_extend(fb, q, torch.device("cpu"), fake_attn)
print(f"\ncp_attn_forward_extend 输出 shape={tuple(out.shape)}  (= [prev*10 | next*10] 拼接)")
print(f"  prev 段 shape={tuple(out[:meta0.total_q_prev_tokens].shape)}, next 段 shape={tuple(out[meta0.total_q_prev_tokens:].shape)}")
print()
print("→ in-seq rank r: Q 分 prev(block_r) / next(block_{2cp-1-r}) 两段,")
print("  各自对不同长度 KV (kv_len_prev < kv_len_next) 算 attention, 再 concat。round-robin 不走这。")

rank 0 (in-seq) zigzag: block 0 + block 7
  total_q_prev=2  total_q_next=2
  actual_seq_q_prev=[2]  actual_seq_q_next=[2]
  kv_len_prev=[2]  (看到 blocks[0..0] = 2 token)
  kv_len_next=[16]  (看到 blocks[0..7] = 16 token)

Q (split, [prev|next]) shape=(4, 4, 6)  (3D: [tokens, heads, qk_head_dim])
  q_prev shape=(2, 4, 6)  (block 0 的 Q)
  q_next shape=(2, 4, 6)  (block 7 的 Q)

  attn_fn #1 (prev): q_seg shape=(2, 4, 6), kv_len=[2], max_q=2
  attn_fn #2 (next): q_seg shape=(2, 4, 6), kv_len=[16], max_q=2

cp_attn_forward_extend 输出 shape=(4, 4, 6)  (= [prev*10 | next*10] 拼接)
  prev 段 shape=(2, 4, 6), next 段 shape=(2, 4, 6)

→ in-seq rank r: Q 分 prev(block_r) / next(block_{2cp-1-r}) 两段,
  各自对不同长度 KV (kv_len_prev < kv_len_next) 算 attention, 再 concat。round-robin 不走这。


---
## 8. 启动期 input_ids 重排（round-robin，EP vs TP）

round-robin 模式下，input_ids 在送入 embedding 前需按 CP 重排，分两种：

**源码**：[cp_utils.py:191](../python/sglang/srt/layers/utils/cp_utils.py#L191)

```python
def cp_round_robin_input_ids(input_ids):
    cp_size = get_attention_cp_size()
    cp_rank = get_attention_cp_rank()
    if get_moe_a2a_backend().is_none():        # TP-only (round-robin-split)
        input_ids = input_ids.reshape(-1, cp_size).T.flatten()   # 全 rank 同序
    else:                                      # DeepEP (in-seq-split)
        input_ids = input_ids[cp_rank::cp_size].contiguous()     # 每 rank 取自己那份
    return input_ids
```

- **a2a none（round-robin-split）**：所有 rank 拿同一份重排后的 input_ids（`reshape(-1,cp).T.flatten()`），再由 `dsa_cp_round_robin_split_data` 切出本 rank token。
- **a2a 非 none（in-seq-split）**：每 rank 直接取 `input_ids[cp_rank::cp_size]`，各 rank 不同。

In [15]:
# 演示 cp_round_robin_input_ids 两种 a2a 模式
# 注: 此函数 V4 路径用 (deepseek_v4.py:1579); V2/GLM 5.2 DSA 不调此函数 (V2 不切 input_ids)
import sglang.srt.layers.moe.utils as moe_utils

set_mode("round-robin-split")
input_ids = torch.arange(SEQ_LEN, dtype=torch.int32)
print(f"原始 input_ids: {input_ids.tolist()}")
print()

# 模式 A: a2a none (round-robin-split 的默认后端)
moe_utils.MOE_A2A_BACKEND = moe_utils.MoeA2ABackend.NONE
cp_utils.get_moe_a2a_backend = lambda: moe_utils.MoeA2ABackend.NONE
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    out = cp_utils.cp_round_robin_input_ids(input_ids)
    print(f"a2a=none  rank {r}: {out.tolist()}  (全 rank 同序, reshape+T 重排)")
print()

# 模式 B: a2a 非 none (in-seq-split 走 DeepEP)
from sglang.srt.layers.moe.utils import MoeA2ABackend
class _FakeA2A:
    def is_none(self): return False   # 模拟 DeepEP
cp_utils.get_moe_a2a_backend = lambda: _FakeA2A()
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    out = cp_utils.cp_round_robin_input_ids(input_ids)
    print(f"a2a=deepep rank {r}: {out.tolist()}  (各 rank 取 input_ids[rank::cp_size])")
print()
print("→ a2a none: 全 rank 同序重排 (配合后续 dsa_cp_round_robin_split_data 切片)")
print("  a2a deepep: 每 rank 直接取自己那份 (各不同)")

原始 input_ids: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

a2a=none  rank 0: [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]  (全 rank 同序, reshape+T 重排)
a2a=none  rank 1: [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]  (全 rank 同序, reshape+T 重排)
a2a=none  rank 2: [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]  (全 rank 同序, reshape+T 重排)
a2a=none  rank 3: [0, 4, 8, 12, 1, 5, 9, 13, 2, 6, 10, 14, 3, 7, 11, 15]  (全 rank 同序, reshape+T 重排)

a2a=deepep rank 0: [0, 4, 8, 12]  (各 rank 取 input_ids[rank::cp_size])
a2a=deepep rank 1: [1, 5, 9, 13]  (各 rank 取 input_ids[rank::cp_size])
a2a=deepep rank 2: [2, 6, 10, 14]  (各 rank 取 input_ids[rank::cp_size])
a2a=deepep rank 3: [3, 7, 11, 15]  (各 rank 取 input_ids[rank::cp_size])

→ a2a none: 全 rank 同序重排 (配合后续 dsa_cp_round_robin_split_data 切片)
  a2a deepep: 每 rank 直接取自己那份 (各不同)


---
## 9. position 拆分（与 data 同构）

position id 也需按 CP 拆分，逻辑与 `cp_split_and_rebuild_data` 完全一致（round-robin stride 切片 / in-seq zigzag），只是作用在 1D position 张量上。

**源码**：[cp_utils.py:167](../python/sglang/srt/layers/utils/cp_utils.py#L167)

In [16]:
# position 拆分: 与 data 同构, 但 1D
positions = torch.arange(SEQ_LEN, dtype=torch.int32)
print(f"原始 positions: {positions.tolist()}")
print()

# round-robin
set_mode("round-robin-split")
print("round-robin position 拆分:")
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    fake_fb = SimpleNamespace(attn_cp_metadata=cp_utils.ContextParallelMetadata())
    p = cp_utils.cp_split_and_rebuild_position(fake_fb, positions)
    print(f"  rank {r}: {p.tolist()}")
print()

# in-seq (用之前 metas)
set_mode("in-seq-split")
print("in-seq position 拆分 (zigzag):")
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    meta = cp_utils.prepare_context_parallel_metadata(
        kv_len=SEQ_LEN, cp_rank=r, cp_size=CP_SIZE,
        seqs_len=[SEQ_LEN], extend_seqs_len=[SEQ_LEN], device="cpu")
    fake_fb = SimpleNamespace(attn_cp_metadata=meta)
    p = cp_utils.cp_split_and_rebuild_position(fake_fb, positions)
    print(f"  rank {r}: {p.tolist()}  (block {meta.zigzag_index[0]} + block {meta.zigzag_index[1]})")

原始 positions: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

round-robin position 拆分:
  rank 0: [0, 4, 8, 12]
  rank 1: [1, 5, 9, 13]
  rank 2: [2, 6, 10, 14]
  rank 3: [3, 7, 11, 15]

in-seq position 拆分 (zigzag):
  rank 0: [0, 1, 14, 15]  (block 0 + block 7)
  rank 1: [2, 3, 12, 13]  (block 1 + block 6)
  rank 2: [4, 5, 10, 11]  (block 2 + block 5)
  rank 3: [6, 7, 8, 9]  (block 3 + block 4)


---
## 10. extend_seq_lens 的 round-robin 切分（多 seq 场景）

多 seq batch 时，`extend_seq_lens`（每 seq 的 extend 长度）需按 round-robin 切成 per-rank q_lens，供 attention metadata 用。

**源码**：[dsa/utils.py:236](../python/sglang/srt/layers/attention/dsa/utils.py#L236) → CPU 实现 `dsa_cp_round_robin_split_q_seqs_cpu` ([dsa/utils.py:221](../python/sglang/srt/layers/attention/dsa/utils.py#L221))

In [17]:
# dsa_cp_round_robin_split_q_seqs_cpu: extend_seq_lens → per-rank q_lens
set_mode("round-robin-split")

# 模拟 3 个 seq, 长度 [8, 4, 6]
extend_seqs = [8, 4, 6]
print(f"extend_seq_lens = {extend_seqs}  (3 个 seq)")
print(f"cp_size = {CP_SIZE}")
print()
for r in range(CP_SIZE):
    global CP_RANK
    CP_RANK = r
    q_lens, bs_idx = dsa_utils.dsa_cp_round_robin_split_q_seqs_cpu(extend_seqs)
    print(f"rank {r}: q_lens={q_lens}  bs_idx={bs_idx}  (本 rank 各 seq 的 Q 长度, 过滤 0)")
print()
print("→ 多 seq 时 round-robin 把每 seq 的 token 按 idx%cp 分给各 rank, 长度不足的 seq 在某些 rank 上为 0 (被过滤)。")

extend_seq_lens = [8, 4, 6]  (3 个 seq)
cp_size = 4

rank 0: q_lens=[2, 1, 2]  bs_idx=[0, 1, 2]  (本 rank 各 seq 的 Q 长度, 过滤 0)
rank 1: q_lens=[2, 1, 2]  bs_idx=[0, 1, 2]  (本 rank 各 seq 的 Q 长度, 过滤 0)
rank 2: q_lens=[2, 1, 1]  bs_idx=[0, 1, 2]  (本 rank 各 seq 的 Q 长度, 过滤 0)
rank 3: q_lens=[2, 1, 1]  bs_idx=[0, 1, 2]  (本 rank 各 seq 的 Q 长度, 过滤 0)

→ 多 seq 时 round-robin 把每 seq 的 token 按 idx%cp 分给各 rank, 长度不足的 seq 在某些 rank 上为 0 (被过滤)。


## 11. 全维度对比总表

| 维度 | round-robin-split | in-seq-split |
|---|---|---|
| 切分粒度 | token 级 (`idx%cp`) | block 级 (`2*cp` 个连续 block) |
| rank r 持有 (hidden_states) | token `r, r+cp, r+2cp, ...` | block_r + block_{2cp-1-r} (首尾配对) |
| 切分函数 | `dsa_cp_round_robin_split_data` | `cp_split_and_rebuild_data` |
| AllGather rerange | `view+transpose+reshape` 三步 | 按 `cp_reverse_index` 重排 |
| 需要 metadata | 否（纯 stride 切片） | 是 (`split_list`/`zigzag_index`/`cp_reverse_index`) |
| KV cache latent (MLA) | `rebuild_cp_kv_cache`, 576 维不展开 | 同左 |
| attention Q 分段 | 否（整段算） | 是, `cp_attn_forward_extend` 分 prev/next |
| MLP 前后通信 | `dsa_cp_gather/reduce_scatter` (AllGather/ReduceScatter) | 旁路 (走 DeepEP A2A) |
| input_ids 重排 | a2a=none, 全 rank 同序 | a2a=deepep, 各 rank 不同 |
| MoE 后端 | TP-only (`ep_size=1`) | DeepEP (`ep_size=tp_size`) |

## 结论

**两种 split 模式的核心差异**：

- **round-robin-split**：token 级轮询（`idx%cp`），切分用 `dsa_cp_round_robin_split_data`（stride 切片），AllGather rerange 用 `view+transpose+reshape` 三步。attention 整段算；MLP 前后加 `dsa_cp_gather/reduce_scatter_hidden_states` 通信；MoE 走 TP-only。
- **in-seq-split**：block 级首尾配对（block_r + block_{2cp-1-r}），切分用 `cp_split_and_rebuild_data` + `prepare_context_parallel_metadata`（zigzag/cp_reverse_index），AllGather rerange 按 `cp_reverse_index` 重排。attention 用 `cp_attn_forward_extend` 分 prev/next 两段算（不同 KV 长度）；MLP 前后旁路，走 DeepEP A2A。

**MLA 的 KV cache 全程不展开**：`rebuild_cp_kv_cache` 在 576 维压缩 latent（kv_lora+rope）上做 AllGather+rerange，靠 weight absorption（W_KC 吸到 Q、W_VC 吸到输出）避免展开成 qk_nope/v 全维（40960），压缩比 71x。

**共同点**：两种模式 AllGather+rerange 后都恢复正序（已 `torch.equal` 验证）；split 态下每 rank 只持 `seq/cp_size` 个 token。